# 07.05 — Target Validation

Run structural, temporal-alignment, binary-label, and anti-leakage validations.

In [2]:
# Import libraries
from pathlib import Path
import sys

In [3]:
# Define the root directory of the project
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CONFIG_PATH = PROJECT_ROOT / 'configs' / 'target_definition.yaml'
CONFIG_PATH

WindowsPath('e:/jcuenca/OneDrive - GUSCanada/5toTerm/01_Capstone/DataLocal/ontario-electricity-peak-risk/configs/target_definition.yaml')

In [4]:
# Import module to manage the Target Definition process
from src.ontario_peak_risk.target_definition.common import (
    load_target_config,
    load_feature_dataset,
    ensure_directories,
)

In [5]:
# Configure the Target Definition process
CONFIG, _ = load_target_config(CONFIG_PATH)
OUTPUT_DIR, REPORTS_DIR, DOCS_DIR = ensure_directories(CONFIG, PROJECT_ROOT)
feature_dataset = load_feature_dataset(CONFIG, PROJECT_ROOT)
feature_dataset.shape

(262944, 100)

In [6]:
# Import modules to build and validate the Forecasting Target and Peak Diagnostics
from src.ontario_peak_risk.target_definition.forecasting_target import build_forecasting_target_matrix
from src.ontario_peak_risk.target_definition.peak_risk_target import build_walk_forward_peak_diagnostics
from src.ontario_peak_risk.target_definition.validation import (
    validate_forecasting_targets,
    validate_peak_diagnostics,
)


In [7]:
# Build the Forecasting Target matrix
FORECAST_CFG = CONFIG['target_definition']['forecasting']
PEAK_CFG = CONFIG['target_definition']['peak_risk']

forecast_matrix = build_forecasting_target_matrix(
    feature_dataset,
    target_column=FORECAST_CFG['target_column'],
    horizon_hours=FORECAST_CFG['horizon_hours'],
    target_prefix=FORECAST_CFG['target_prefix'],
)
forecast_matrix.shape

(262944, 28)

In [8]:
# Display the first few rows of the forecasting target matrix
forecast_matrix.head()

,fsa,forecast_origin,target_h01,target_h02,target_h03,target_h04,target_h05,target_h06,target_h07,target_h08,...,target_h17,target_h18,target_h19,target_h20,target_h21,target_h22,target_h23,target_h24,available_horizons,complete_24h_target
0,L4T,2021-01-01 00:00:00,9585.1,9015.5,8593.1,8353.5,8334.4,8499.5,8784.2,9313.2,...,13737.4,13772.0,13490.4,13255.0,12622.0,11607.1,10571.4,9630.6,24,1
1,L4T,2021-01-01 01:00:00,9015.5,8593.1,8353.5,8334.4,8499.5,8784.2,9313.2,10062.2,...,13772.0,13490.4,13255.0,12622.0,11607.1,10571.4,9630.6,8984.5,24,1
2,L4T,2021-01-01 02:00:00,8593.1,8353.5,8334.4,8499.5,8784.2,9313.2,10062.2,10968.0,...,13490.4,13255.0,12622.0,11607.1,10571.4,9630.6,8984.5,8434.7,24,1
3,L4T,2021-01-01 03:00:00,8353.5,8334.4,8499.5,8784.2,9313.2,10062.2,10968.0,11971.4,...,13255.0,12622.0,11607.1,10571.4,9630.6,8984.5,8434.7,8151.9,24,1
4,L4T,2021-01-01 04:00:00,8334.4,8499.5,8784.2,9313.2,10062.2,10968.0,11971.4,12425.2,...,12622.0,11607.1,10571.4,9630.6,8984.5,8434.7,8151.9,8059.8,24,1


In [9]:
# Build the Walk-Forward Peak Diagnostics
peak_labels, peak_thresholds = build_walk_forward_peak_diagnostics(
    feature_dataset,
    percentile=PEAK_CFG['percentile_threshold'],
    target_column=PEAK_CFG['target_column'],
    grouping_columns=PEAK_CFG['grouping_columns'],
    time_column=PEAK_CFG['diagnostic_time_column'],
    minimum_training_years=PEAK_CFG['minimum_training_years'],
)
peak_labels.shape, peak_thresholds.shape

((210384, 8), (96, 6))

In [10]:
# Display the first few rows of the peak labels
peak_labels.head()

,fsa,timestamp,year,season,total_consumption_kwh,peak_threshold_kwh,peak_threshold_available,peak_risk_target
0,L4T,2022-01-01 00:00:00,2022,Winter,9544.8,15253.3325,1,0
1,L4T,2022-01-01 01:00:00,2022,Winter,8903.8,15253.3325,1,0
2,L4T,2022-01-01 02:00:00,2022,Winter,8281.0,15253.3325,1,0
3,L4T,2022-01-01 03:00:00,2022,Winter,7882.5,15253.3325,1,0
4,L4T,2022-01-01 04:00:00,2022,Winter,7676.0,15253.3325,1,0


In [11]:
# Display the first few rows of the peak thresholds
peak_thresholds.head()

,fsa,season,peak_threshold_kwh,evaluation_year,training_year_start,training_year_end
0,L4T,Fall,14434.4425,2022,2021,2021
1,L4T,Spring,14997.0125,2022,2021,2021
2,L4T,Summer,23769.5000,2022,2021,2021
3,L4T,Winter,15253.3325,2022,2021,2021
4,M5R,Fall,9531.8625,2022,2021,2021


In [12]:
# Validate the Forecasting Target matrix
forecast_validation = validate_forecasting_targets(
    feature_dataset,
    forecast_matrix,
    target_column=FORECAST_CFG['target_column'],
    horizon_hours=FORECAST_CFG['horizon_hours'],
)
forecast_validation.shape

(26, 4)

In [13]:
# Display the first few rows of the forecast validation results
forecast_validation

,check,violations,status,description
0,target_origin_row_count,0,PASS,Target matrix must preserve one forecast origi...
1,unique_forecast_origin,0,PASS,FSA + forecast origin must remain unique.
2,alignment_h01,0,PASS,h+1 target must equal observed consumption exa...
3,alignment_h02,0,PASS,h+2 target must equal observed consumption exa...
4,alignment_h03,0,PASS,h+3 target must equal observed consumption exa...
5,alignment_h04,0,PASS,h+4 target must equal observed consumption exa...
6,alignment_h05,0,PASS,h+5 target must equal observed consumption exa...
7,alignment_h06,0,PASS,h+6 target must equal observed consumption exa...
8,alignment_h07,0,PASS,h+7 target must equal observed consumption exa...
9,alignment_h08,0,PASS,h+8 target must equal observed consumption exa...


In [14]:
# Validate the Walk-Forward Peak Diagnostics
peak_validation = validate_peak_diagnostics(
    peak_labels,
    peak_thresholds,
)
peak_validation.shape

(4, 4)

In [15]:
# Display the first few rows of the peak validation results
peak_validation


,check,violations,status,description
0,unique_peak_label_key,0,PASS,Diagnostic Peak labels must be unique by FSA +...
1,peak_label_binary,0,PASS,Peak-Risk labels must be binary when a thresho...
2,peak_threshold_availability_consistency,0,PASS,Available thresholds must produce a diagnostic...
3,walk_forward_threshold_no_future_year,0,PASS,Each diagnostic threshold must be fitted using...


In [17]:
# Check that all validation checks passed
assert (forecast_validation['status'] == 'PASS').all()
assert (peak_validation['status'] == 'PASS').all()
print('All Target Definition Phase target validation checks passed.')

All Target Definition Phase target validation checks passed.
